# 4-4. 커스텀 LLM 서비스: FastAPI + Gradio

⏱ **소요시간**: 30분

## 학습 목표

- FastAPI로 `/chat` SSE(Server-Sent Events) 스트리밍 엔드포인트를 구현한다.
- 간단한 **Bearer Token** 인증 미들웨어를 추가한다.
- OpenAI 호환 `/v1/chat/completions` 프록시 패턴으로 기존 OpenAI SDK 클라이언트를 그대로 활용한다.
- Gradio `ChatInterface`로 프런트엔드를 만들고 FastAPI 백엔드를 호출한다.
- 노트북 안에서 uvicorn을 `nest_asyncio` + 백그라운드 스레드로 기동하는 방법을 익힌다.

## 핵심 키워드

`FastAPI` · `SSE` · `StreamingResponse` · `LangChain stream` · `OpenAI 호환 API` · `Bearer token` · `Gradio` · `ChatInterface` · `uvicorn` · `nest_asyncio`

## 아키텍처

```
Gradio UI (:7860)  ─┐
OpenAI SDK 클라이언트 ─┼─▶  FastAPI (:8000)  ─▶  common.get_llm()  ─▶  🔒 Ollama  or  ☁️ OpenAI
curl / 타 서비스     ─┘
```

> 💡 폐쇄망에서는 FastAPI 앱도 DMZ의 WAF/nginx 뒤에 두고, 내부 LLM은 localhost에서만 바인딩한다.

In [ ]:
# 공통 헬퍼
import sys
sys.path.insert(0, '../..')
from common import get_llm, provider_badge

print(provider_badge())

## 1. FastAPI `/chat` SSE 스트리밍 엔드포인트

- LangChain `ChatModel.stream()` → 토큰별 `AIMessageChunk` 반복
- SSE는 `data: <payload>\n\n` 형태의 단방향 스트림. HTTP/1.1 유지 연결.
- `media_type="text/event-stream"` 으로 응답.

In [ ]:
import json, os, time
from typing import AsyncIterator

from fastapi import FastAPI, Depends, HTTPException, Request, status
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

app = FastAPI(title="Airgap LLM Gateway", version="0.1.0")

# --- 인증 ---
security = HTTPBearer(auto_error=True)
ALLOWED_TOKENS = {os.getenv("API_TOKEN", "dev-secret-please-change")}


def require_token(creds: HTTPAuthorizationCredentials = Depends(security)):
    if creds.credentials not in ALLOWED_TOKENS:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="invalid token")
    return creds.credentials


# --- /chat SSE ---
class ChatRequest(BaseModel):
    message: str
    system: str | None = "당신은 폐쇄망 금융 도우미입니다. 간결히 한국어로 답하세요."


async def sse_stream(req: ChatRequest) -> AsyncIterator[bytes]:
    llm = get_llm(temperature=0.2, streaming=True)
    messages = [SystemMessage(content=req.system or ""), HumanMessage(content=req.message)]
    try:
        for chunk in llm.stream(messages):
            text = chunk.content if isinstance(chunk.content, str) else str(chunk.content)
            if text:
                yield f"data: {json.dumps({'delta': text}, ensure_ascii=False)}\n\n".encode("utf-8")
        yield b"data: [DONE]\n\n"
    except Exception as e:
        yield f"data: {json.dumps({'error': str(e)})}\n\n".encode("utf-8")


@app.post("/chat")
async def chat(req: ChatRequest, _: str = Depends(require_token)):
    return StreamingResponse(sse_stream(req), media_type="text/event-stream")


@app.get("/healthz")
def healthz():
    return {"status": "ok", "provider": os.getenv("LLM_PROVIDER", "openai")}

## 2. OpenAI 호환 `/v1/chat/completions` 프록시

많은 도구(OpenAI SDK, LangChain, LlamaIndex, cursor/continue 등)가 `/v1/chat/completions` 포맷을 지원한다.
같은 엔드포인트를 우리 게이트웨이에서 흉내내면, **클라이언트 코드 수정 없이** `OPENAI_BASE_URL`만 바꿔서 폐쇄망 LLM을 사용할 수 있다.

- 요청 스키마: `{model, messages:[{role,content}], stream, temperature, ...}`
- 응답 스키마: 비스트리밍 `ChatCompletion`, 스트리밍 `ChatCompletionChunk` (SSE)
- 주의: **완벽 호환은 난이도 높음** (function calling, tool_choice, logprobs 등). 본 예제는 **텍스트 메시지만** 지원하는 최소 호환.

In [ ]:
import uuid
from typing import Literal

class OAIMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class OAIChatRequest(BaseModel):
    model: str = "airgap-default"
    messages: list[OAIMessage]
    stream: bool = False
    temperature: float = 0.2


def _to_lc_messages(messages: list[OAIMessage]):
    mapping = {"system": SystemMessage, "user": HumanMessage, "assistant": AIMessage}
    return [mapping[m.role](content=m.content) for m in messages]


def _oai_chunk(model: str, delta_content: str, finish_reason: str | None = None) -> dict:
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [
            {
                "index": 0,
                "delta": {"content": delta_content} if delta_content else {},
                "finish_reason": finish_reason,
            }
        ],
    }


async def _oai_stream(req: OAIChatRequest) -> AsyncIterator[bytes]:
    llm = get_llm(temperature=req.temperature, streaming=True)
    for chunk in llm.stream(_to_lc_messages(req.messages)):
        text = chunk.content if isinstance(chunk.content, str) else str(chunk.content)
        if text:
            payload = _oai_chunk(req.model, text)
            yield f"data: {json.dumps(payload, ensure_ascii=False)}\n\n".encode("utf-8")
    yield f"data: {json.dumps(_oai_chunk(req.model, '', finish_reason='stop'))}\n\n".encode("utf-8")
    yield b"data: [DONE]\n\n"


@app.post("/v1/chat/completions")
async def openai_compat(req: OAIChatRequest, _: str = Depends(require_token)):
    if req.stream:
        return StreamingResponse(_oai_stream(req), media_type="text/event-stream")

    llm = get_llm(temperature=req.temperature, streaming=False)
    msg = llm.invoke(_to_lc_messages(req.messages))
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    return JSONResponse({
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": content},
            "finish_reason": "stop",
        }],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    })

## 3. 노트북에서 uvicorn 기동 (백그라운드 스레드)

> 실무에서는 별도 터미널에서 `uvicorn app:app --host 0.0.0.0 --port 8000`을 띄우는 것이 안정적이다.
> 노트북 안에서 동작 확인만 할 때 아래 패턴을 사용한다.

In [ ]:
import threading, time

import nest_asyncio, uvicorn
nest_asyncio.apply()

def _run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(1.5)
print("FastAPI 기동: http://127.0.0.1:8000  (Bearer token:", list(ALLOWED_TOKENS)[0], ")")

In [ ]:
# 헬스체크
import httpx

print(httpx.get("http://127.0.0.1:8000/healthz").json())

In [ ]:
# /chat SSE 스트리밍 호출
token = list(ALLOWED_TOKENS)[0]
with httpx.stream(
    "POST",
    "http://127.0.0.1:8000/chat",
    headers={"Authorization": f"Bearer {token}"},
    json={"message": "폐쇄망 LLM 도입의 장단점을 3가지씩 요약해줘"},
    timeout=60.0,
) as r:
    for line in r.iter_lines():
        if line.startswith("data: "):
            data = line[6:]
            if data == "[DONE]":
                break
            try:
                print(json.loads(data).get("delta", ""), end="", flush=True)
            except json.JSONDecodeError:
                pass

In [ ]:
# OpenAI SDK로 /v1/chat/completions 호출
# pip install openai
try:
    from openai import OpenAI
    client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key=token)
    resp = client.chat.completions.create(
        model="airgap-default",
        messages=[{"role": "user", "content": "청약철회 기간은 며칠인지 한 줄로"}],
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print("[스킵]", type(e).__name__, str(e)[:200])

## 4. Gradio ChatInterface 프런트엔드

Gradio는 내부 데모/PoC에 가장 빠른 UI 프레임워크. `ChatInterface`는 메시지 스트리밍(generator)을 지원한다.

> 아래 셀은 실행 시 `http://127.0.0.1:7860`에 UI를 띄운다. 교실 환경에 따라 생략 가능.

In [ ]:
import gradio as gr

API_URL = "http://127.0.0.1:8000/chat"

def respond(message: str, history: list):
    """Gradio generator: 백엔드 SSE를 받아 토큰별로 yield."""
    partial = ""
    try:
        with httpx.stream(
            "POST", API_URL,
            headers={"Authorization": f"Bearer {token}"},
            json={"message": message},
            timeout=120.0,
        ) as r:
            for line in r.iter_lines():
                if line.startswith("data: "):
                    data = line[6:]
                    if data == "[DONE]":
                        break
                    try:
                        delta = json.loads(data).get("delta", "")
                    except json.JSONDecodeError:
                        delta = ""
                    partial += delta
                    yield partial
    except Exception as e:
        yield f"[에러] {e}"

demo = gr.ChatInterface(
    fn=respond,
    title="🔒 Airgap LLM Chat",
    description=f"Provider: {provider_badge()}",
    examples=["청약철회 기간은?", "개인정보보호법상 가명정보란?", "전자금융감독규정 §14 요약"],
)

# 실제 교실에서는 아래 주석 해제해 UI 띄움
# demo.launch(server_name="127.0.0.1", server_port=7860, share=False, inbrowser=False)
print("Gradio demo 객체 준비 완료 — demo.launch() 로 기동")

## 5. 운영 체크리스트

- **인증**: Bearer → **SSO/OIDC + mTLS**로 승격. 토큰은 Vault/KMS에 저장.
- **감사**: 모든 요청/응답을 Session 4-2의 `audit_log()` 유틸로 기록.
- **Rate limit**: `slowapi`, `nginx limit_req`, API Gateway로 DoS/LLM04 대응.
- **Timeout & Backpressure**: `uvicorn --timeout-keep-alive`, LLM `max_tokens` 상한.
- **Observability**: OpenTelemetry (`opentelemetry-instrumentation-fastapi`) + Prometheus 메트릭.
- **배포**: `gunicorn -k uvicorn.workers.UvicornWorker -w 4 app:app` 또는 K8s Deployment.
